# How InfoCars priced used cars, step by step

**The question a dealer asks.** "What is this car worth, and if I price it here, how long will it sit?" InfoCars
answered it for used-car dealers in Brazil: it collected ads from sale websites, matched each one to a standard make,
model, version and year, and showed what comparable cars were listed for and how long they took to sell.

This notebook rebuilds that pipeline on the repository's **1,200 synthetic ads**, in the order it ran:

| Step | What | Code |
|---|---|---|
| 1 | Look at the raw ads and their mistakes | pandas |
| 2 | Match free-text titles to a catalogue | [`infocars/normalize.py`](../infocars/normalize.py) |
| 3 | Drop bad prices and outliers | pandas, as [`infocars/db.py`](../infocars/db.py) |
| 4 | Price every ad against its comparables | pandas |
| 5 | Does price position change time to sell? | pandas + Altair |
| 6 | Let a language model query the data safely | [`infocars/sql_guard.py`](../infocars/sql_guard.py) |
| 7 | Reconcile with the Python demo and the dbt marts | pandas + DuckDB |

The production figures (about 250,000 listings, 5,114 model spellings reduced to 641 models, about R$ 2,000 saved per
negotiation) are in the [README](../README.md); none of the numbers below are results of the real business.

**The charts are interactive**: click a price band, brush the scatter, pick a car from the menu. They are built with
[Altair](https://altair-viz.github.io/); open the [interactive version](https://guilhermerigonato.com/infocars/notebook/)
to use them (GitHub shows notebooks without JavaScript).

**Run it:** `pip install -r requirements.txt pandas altair duckdb dbt-duckdb`, build the dbt model once
(`cd dashboards && dbt build --profiles-dir .`) and run all cells (a few seconds).

In [1]:
import sys
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import altair as alt

sys.path.insert(0, "..")
from infocars.normalize import match_listing, clean
from infocars.db import build_database, market_price, MIN_PRICE, MAX_PRICE, SNAPSHOT_DATE
from infocars.sql_guard import check_sql, SqlGuardError

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_colwidth", 60)
BLUE, ORANGE, GREEN, GREY, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#c9c8c3", "#52514e", "#e4e3df"

@alt.theme.register("portfolio", enable=True)
def portfolio_theme():
    axis = {"gridColor": GRID, "domainColor": MUTED, "tickColor": MUTED, "labelColor": MUTED,
            "titleColor": MUTED, "labelFontSize": 11, "titleFontSize": 11, "titleFontWeight": "normal"}
    return {"config": {"view": {"stroke": None}, "axis": axis, "legend": {"labelColor": MUTED, "titleColor": MUTED},
                       "title": {"anchor": "start", "fontSize": 13, "color": "#0b0b0b"},
                       "font": "system-ui, -apple-system, Segoe UI, sans-serif"}}

## Step 1. The raw ads

[`scripts/generate_sample_data.py`](../scripts/generate_sample_data.py) writes ads the way sellers write them: mixed
case, model years as "22/23", versions in any order, typos, missing prices and prices typed without their last digit.
It also makes cars priced below the market leave the site faster, so step 5 can check that the analysis recovers
a pattern that was put there on purpose.

In [2]:
raw = pd.read_csv("../data/sample_listings.csv", parse_dates=["listed_at", "removed_at"])
print(f"{len(raw):,} ads from {raw['source'].nunique()} sites, listed {raw['listed_at'].min():%d %b %Y} to {raw['listed_at'].max():%d %b %Y}")
raw.sample(8, random_state=3)[["title", "price_brl", "km", "state", "listed_at", "removed_at"]]

1,200 ads from 3 sites, listed 01 May 2025 to 18 Oct 2025


,title,price_brl,km,state,listed_at,removed_at
822,Fiat Argo Trekking 1.0 Flex 2024,"8,476.00",12422,BA,2025-06-15,NaT
448,JEEP COMPASS 1.0 Longitude 2021,"108,100.00",44391,GO,2025-06-05,2025-08-26
806,Chevy ONIX PREMIER Automático 2023,"79,900.00",22650,SP,2025-09-03,2025-10-21
415,TOYOTA COROLLA GLI 2021,"88,600.00",47773,SC,2025-06-07,2025-06-28
290,HYUNDAI hb20 Sense Único dono 2022,"51,400.00",38995,RJ,2025-09-17,2025-10-10
619,Chevrolet onix PREMIERE 1.0 Flex 23/24,"94,700.00",8494,SP,2025-10-03,NaT
1117,COMPASS LIMITED 1.0 Turbo 2019,"101,200.00",62177,SP,2025-05-08,2025-05-26
343,JEEP Compas Completo LONGITUDE 21/22,"115,000.00",31529,SP,2025-06-30,2025-08-16


In [3]:
spelling = raw["title"].str.lower().str.split().str[:2].str.join(" ")
print(f"{spelling.nunique()} different ways of writing make and model in the first two words, for example:")
spelling[spelling.str.contains("onix|corol", regex=True)].value_counts().head(12)

116 different ways of writing make and model in the first two words, for example:


title
toyota corolla    104
chevrolet onix     75
toyota corola      35
chevy onix         30
gm onix            28
corolla altis      11
corolla gli         9
corolla 1.0         8
onix 1.0            8
corolla xei         5
onix ls             4
onix ltz            4
Name: count, dtype: int64

## Step 2. Match each title to the catalogue

`match_listing()` cleans the text (lowercase, no accents, no punctuation), finds make, model and version by alias on
whole words (so "LT" never matches "LTZ"), reads the year from forms such as "22/23" (the second number is the model
year), and only then tries a fuzzy match for typos such as "Corola" or "Platinun". Short tokens are never
fuzzy-matched, because "lt" and "ltz" are one letter apart.

In [4]:
examples = ["GM ONIX LT 22/23", "onix ltz 1.0 turbo 2023", "Toyota Corola XEi 2021", "Hyundai HB20 Platinun 2022",
            "Chevrolet Onix 1.0 LT Flex Manual 2022"]
pd.DataFrame([dict(title=t, cleaned=clean(t), **match_listing(t).__dict__) for t in examples])

,title,cleaned,make,model,version,year,method
0,GM ONIX LT 22/23,gm onix lt 22/23,chevrolet,onix,lt,2023,exact
1,onix ltz 1.0 turbo 2023,onix ltz 1.0 turbo 2023,chevrolet,onix,ltz,2023,exact
2,Toyota Corola XEi 2021,toyota corola xei 2021,toyota,corolla,xei,2021,fuzzy
3,Hyundai HB20 Platinun 2022,hyundai hb20 platinun 2022,hyundai,hb20,platinum,2022,fuzzy
4,Chevrolet Onix 1.0 LT Flex Manual 2022,chevrolet onix 1.0 lt flex manual 2022,chevrolet,onix,lt,2022,exact


In [5]:
m = pd.DataFrame([match_listing(t).__dict__ for t in raw["title"]])
ads = pd.concat([raw, m], axis=1)
ads["complete"] = ads[["make", "model", "version", "year"]].notna().all(axis=1)
print(f"Complete matches: {ads['complete'].sum():,} of {len(ads):,}; through typo correction: {(ads['method'] == 'fuzzy').sum()}")
print(f"{spelling.nunique()} spellings → {ads.loc[ads['complete'], ['make', 'model']].drop_duplicates().shape[0]} models")

Complete matches: 1,200 of 1,200; through typo correction: 190


116 spellings → 7 models


## Step 3. Quality checks

In the same order as the production checks ([`sql/postgres/03_quality_checks.sql`](../sql/postgres/03_quality_checks.sql)):

1. **Bad price:** missing, or outside R$ 5,000–1,500,000.
2. **No full match:** make, model, version or year missing.
3. **Outlier:** more than 2× or less than half the median of the same model, version and year (groups of 3 or more).
   This catches "R$ 15,715" typed for a R$ 157,150 Corolla, which still looks like a car price.

In [6]:
ads["status"] = "kept"
ads.loc[ads["price_brl"].isna() | ~ads["price_brl"].between(MIN_PRICE, MAX_PRICE), "status"] = "bad price"
ads.loc[(ads["status"] == "kept") & ~ads["complete"], "status"] = "no full match"
k = ads["status"] == "kept"
grp = ads[k].groupby(["model", "version", "year"])["price_brl"]
peer_n, peer_med = grp.transform("size"), grp.transform("median")
outlier = (peer_n >= 3) & ~((ads.loc[k, "price_brl"] >= 0.5 * peer_med) & (ads.loc[k, "price_brl"] <= 2 * peer_med))
ads.loc[outlier[outlier].index, "status"] = "outlier"

flow = ads["status"].value_counts().reindex(["kept", "bad price", "no full match", "outlier"]).fillna(0).astype(int)
print(flow.to_string())
ads[ads["status"] == "outlier"][["title", "price_brl"]].assign(peer_median=peer_med).head(5)

status
kept             1167
bad price          15
no full match       0
outlier            18


,title,price_brl,peer_median
1,TOYOTA Corolla ALTIS 1.0 Flex 2024,"15,715.00","151,500.00"
226,TOYOTA Corolla Flex ALTIS 2022,"11,470.00","122,000.00"
281,Volks POLO Completo Comfortline 24/25,"8,720.00","93,900.00"
297,Hyundai hb20 Manual PLATINUM 2022,"7,390.00","72,800.00"
305,Volkswagen POLO HIGHLINE 2023,"8,526.00","86,500.00"


## Step 4. Price every ad against its comparables

An ad's market is the other clean ads of the same version and model year (in all states), so an ad is never its own
comparable. With fewer than 5 comparables there is no market, as the assistant calls such an estimate weak. Position
follows `price_position()`: above the 75th percentile is *above market*, below the 25th is *below market*. Days on the
market: an ad that disappears counts as sold on its last day online; ads still online are measured to the snapshot
date (1 November 2025).

In [7]:
clean_ads = ads[ads["status"] == "kept"].copy()
snapshot = pd.Timestamp(SNAPSHOT_DATE)
clean_ads["is_sold"] = clean_ads["removed_at"].notna()
clean_ads["days_on_market"] = (clean_ads["removed_at"].fillna(snapshot) - clean_ads["listed_at"]).dt.days

def market(group):
    out = []
    p = group["price_brl"].to_numpy()
    for i in range(len(group)):
        others = np.delete(p, i)
        if len(others) >= 5:
            out.append((len(others), *np.round(np.quantile(others, [0.25, 0.5, 0.75]))))
        else:
            out.append((len(others), np.nan, np.nan, np.nan))
    return pd.DataFrame(out, index=group.index, columns=["comparables", "p25", "median", "p75"])

clean_ads = clean_ads.join(clean_ads.groupby(["model", "version", "year"], group_keys=False).apply(market))
clean_ads["gap_pct"] = clean_ads["price_brl"] / clean_ads["median"] - 1
clean_ads["position"] = np.select(
    [clean_ads["median"].isna(), clean_ads["price_brl"] > clean_ads["p75"], clean_ads["price_brl"] < clean_ads["p25"]],
    ["too few comparables", "above market", "below market"], "within market range")
BANDS = ["Below -6%", "-6% to -2%", "-2% to +2%", "+2% to +6%", "Above +6%"]
g = clean_ads["gap_pct"]
clean_ads["band"] = np.select([g.isna(), g < -0.06, g < -0.02, g <= 0.02, g <= 0.06],
                              ["Too few comparables"] + BANDS[:4], BANDS[4])
clean_ads["car"] = (clean_ads["make"].str.title() + " " + clean_ads["model"].str.title() + " "
                    + clean_ads["version"].str.upper() + " " + clean_ads["year"].astype(int).astype(str))
clean_ads[["title", "price_brl", "comparables", "p25", "median", "p75", "gap_pct", "position", "days_on_market"]].head(6)

,title,price_brl,comparables,p25,median,p75,gap_pct,position,days_on_market
0,Toyota Corolla GLI 18/19,"72,900.00",10,"65,375.00","67,700.00","68,225.00",0.08,above market,5
2,hb20 1.0 Vision 2023,"63,400.00",4,NaN,NaN,NaN,NaN,too few comparables,27
3,HYUNDAI HB20 Sense Flex 24/25,"66,300.00",3,NaN,NaN,NaN,NaN,too few comparables,14
4,hb20 COMFORT Completo 2024,"67,000.00",8,"68,425.00","72,300.00","73,400.00",-0.07,below market,11
5,Chevrolet Tracker PREMIER 2023,"113,400.00",7,"106,350.00","110,000.00","112,300.00",0.03,above market,33
6,Volks Polo 1.0 GTS 22/23,"97,900.00",7,"100,850.00","102,300.00","107,150.00",-0.04,below market,46


## Step 5. Does price position change time to sell?

**Click a price band** on the left to see how long its cars took to sell on the right (sold ads only). Cars priced
under the market should leave faster: that is the pattern the generator planted.

In [8]:
priced = clean_ads[clean_ads["band"] != "Too few comparables"]
sold = priced[priced["is_sold"]]
bands = sold.groupby("band")["days_on_market"].agg(median_days="median", sold="size").reindex(BANDS).reset_index()
bands["sold_within_30d"] = sold.groupby("band")["days_on_market"].apply(lambda d: (d <= 30).mean()).reindex(BANDS).to_numpy()
bands["still_online"] = priced.groupby("band")["is_sold"].apply(lambda s: 1 - s.mean()).reindex(BANDS).to_numpy()

pick = alt.selection_point(name="band", fields=["band"], on="click", value=[{"band": "Below -6%"}])
band_colors = alt.Scale(domain=BANDS, range=[GREEN, "#7fcfae", GREY, "#f2b08f", ORANGE])
left = alt.Chart(bands).mark_bar(cornerRadius=3).encode(
    x=alt.X("band:N", sort=BANDS, title="Asking price vs comparable median", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("median_days:Q", title="Median days to sell"),
    color=alt.Color("band:N", scale=band_colors, legend=None),
    opacity=alt.condition(pick, alt.value(1), alt.value(0.35)),
    tooltip=["band", "median_days", "sold", alt.Tooltip("sold_within_30d:Q", format=".0%"), alt.Tooltip("still_online:Q", format=".0%")],
).add_params(pick).properties(width=330, height=230, title="Cars priced under the market sell faster (click a band)")
right = alt.Chart(sold[["band", "days_on_market"]]).transform_filter(pick).mark_bar(color=BLUE).encode(
    x=alt.X("days_on_market:Q", bin=alt.Bin(step=10), title="Days to sell"),
    y=alt.Y("count():Q", title="Sold ads"),
    color=alt.Color("band:N", scale=band_colors, legend=None),
).properties(width=300, height=230, title=alt.Title(alt.expr("'Days to sell · ' + (band.band ? band.band[0] : '')")))
alt.hconcat(left, right)

alt.HConcatChart(...)

The same in one view, ad by ad. **Drag a box** on the scatter to select ads; the bars below count them by position.

In [9]:
pts = priced[["car", "state", "price_brl", "median", "gap_pct", "days_on_market", "is_sold", "position"]].copy()
pts["status"] = np.where(pts["is_sold"], "Sold", "Still online")
box = alt.selection_interval()
pos_colors = alt.Scale(domain=["below market", "within market range", "above market"], range=[GREEN, GREY, ORANGE])
scatter = alt.Chart(pts).mark_circle(size=40, opacity=0.7).encode(
    x=alt.X("gap_pct:Q", title="Asking price vs comparable median", axis=alt.Axis(format="+%")),
    y=alt.Y("days_on_market:Q", title="Days on the market"),
    color=alt.condition(box, alt.Color("position:N", scale=pos_colors, title="Position"), alt.value("#e6e6e6")),
    shape=alt.Shape("status:N", title=None),
    tooltip=["car", "state", alt.Tooltip("price_brl:Q", format=",.0f"), alt.Tooltip("median:Q", format=",.0f"),
             alt.Tooltip("gap_pct:Q", format="+.1%"), "days_on_market", "status"],
).add_params(box).properties(width=560, height=260, title="Every priced ad (drag a box)")
counts = alt.Chart(pts).transform_filter(box).mark_bar().encode(
    y=alt.Y("position:N", title=None, sort=["below market", "within market range", "above market"]),
    x=alt.X("count():Q", title="Ads in the box"),
    color=alt.Color("position:N", scale=pos_colors, legend=None),
).properties(width=560, height=90)
alt.vconcat(scatter, counts)

alt.VConcatChart(...)

And what a dealer saw for one car: **pick a car** from the menu under the chart. Each dot is a comparable ad; the band
is the 25th–75th percentile range and the line the median.

In [10]:
cars = priced.groupby("car").filter(lambda d: len(d) >= 12)
car_names = sorted(cars["car"].unique())
choose = alt.param(name="car_pick", value="Chevrolet Onix LTZ 2023" if "Chevrolet Onix LTZ 2023" in car_names else car_names[0],
                   bind=alt.binding_select(options=car_names, name="Car: "))
base = alt.Chart(cars).transform_filter("datum.car == car_pick")
band = base.transform_aggregate(p25="q1(price_brl)", p75="q3(price_brl)", med="median(price_brl)").mark_rect(
    opacity=0.15, color=BLUE).encode(x="p25:Q", x2="p75:Q")
med = base.transform_aggregate(med="median(price_brl)").mark_rule(color=BLUE, strokeWidth=2).encode(x="med:Q")
dots = base.mark_tick(thickness=2, size=40).encode(
    x=alt.X("price_brl:Q", title="Asking price (R$)", scale=alt.Scale(zero=False), axis=alt.Axis(format=",.0f")),
    color=alt.Color("status:N", scale=alt.Scale(domain=["Sold", "Still online"], range=[MUTED, ORANGE]), title=None),
    tooltip=["state", alt.Tooltip("price_brl:Q", format=",.0f"), "days_on_market", "status"],
).transform_calculate(status="datum.is_sold ? 'Sold' : 'Still online'")
(band + med + dots).add_params(choose).properties(width=560, height=90, title="Comparable ads for the chosen car")

alt.LayerChart(...)

## Step 6. Let a language model query the data safely

The pricing assistant ([`infocars/assistant.py`](../infocars/assistant.py)) uses the Claude API with tool use: two
fixed tools (`market_price`, `price_position`) and a guarded `run_sql` for questions the fixed tools do not cover.
Anything the model writes goes through `check_sql()` before it runs: one SELECT only, approved tables only, no
comments, a row limit added. In production the query also ran under a read-only database role, so it had to pass both.

In [11]:
tries = [
    "SELECT model, version, AVG(price_brl) FROM listings GROUP BY 1, 2",
    "select * from market_by_version where year = 2023 limit 20",
    "WITH s AS (SELECT * FROM listings WHERE state = 'SP') SELECT COUNT(*) FROM s",
    "DELETE FROM listings",
    "SELECT * FROM listings; DROP TABLE listings",
    "SELECT * FROM raw_listings",
    "SELECT * FROM listings -- ignore the rules",
]
rows = []
for q in tries:
    try:
        rows.append((q, "allowed", check_sql(q)))
    except SqlGuardError as e:
        rows.append((q, "refused", str(e)))
pd.DataFrame(rows, columns=["query written by the model", "result", "query run / reason"])

,query written by the model,result,query run / reason
0,"SELECT model, version, AVG(price_brl) FROM listings GROU...",allowed,"SELECT model, version, AVG(price_brl) FROM listings GROU..."
1,select * from market_by_version where year = 2023 limit 20,allowed,select * from market_by_version where year = 2023 limit 20
2,WITH s AS (SELECT * FROM listings WHERE state = 'SP') SE...,allowed,WITH s AS (SELECT * FROM listings WHERE state = 'SP') SE...
3,DELETE FROM listings,refused,Only SELECT queries are allowed.
4,SELECT * FROM listings; DROP TABLE listings,refused,Only one statement is allowed.
5,SELECT * FROM raw_listings,refused,Table not allowed: raw_listings.
6,SELECT * FROM listings -- ignore the rules,refused,Comments are not allowed.


## Step 7. Reconcile with the Python demo and the dbt marts

Three implementations must agree: the SQLite demo in `infocars/db.py`, the dbt model in [`dashboards/`](../dashboards)
and this notebook.

In [12]:
conn, report = build_database("../data/sample_listings.csv")
demo = market_price(conn, "onix", "ltz", 2023)
db = duckdb.connect("../dashboards/infocars.duckdb", read_only=True)
m_band = db.sql("select price_band, median_days_to_sell, listings from mart_price_position").df().set_index("price_band")
m_out = db.sql("select price_position, count(*) n, sum(gap_to_median_brl) gap from mart_stock_out_of_line group by 1").df().set_index("price_position")
db.close()

online = clean_ads[~clean_ads["is_sold"]]
above = online[online["position"] == "above market"]
onix = clean_ads[(clean_ads["model"] == "onix") & (clean_ads["version"] == "ltz") & (clean_ads["year"] == 2023)]
recon = pd.DataFrame([
    ["Ads kept", flow["kept"], report["matched"], None],
    ["Dropped: bad price", flow["bad price"], report["bad_price"], None],
    ["Dropped: outliers", flow["outlier"], report["outliers"], None],
    ["Matched through typo correction", (clean_ads["method"] == "fuzzy").sum(), report["fuzzy"], None],
    ["2023 Onix LTZ median price (R$)", onix["price_brl"].median(), demo["median_price_brl"], None],
    ["Median days to sell, below −6%", bands.set_index("band").loc["Below -6%", "median_days"], None, m_band.loc["Below -6%", "median_days_to_sell"]],
    ["Median days to sell, above +6%", bands.set_index("band").loc["Above +6%", "median_days"], None, m_band.loc["Above +6%", "median_days_to_sell"]],
    ["Ads online above market", len(above), None, m_out.loc["above market", "n"]],
    ["… total above their medians (R$)", (above["price_brl"] - above["median"]).sum(), None, m_out.loc["above market", "gap"]],
], columns=["measure", "notebook", "Python demo", "dbt"])
recon["match"] = [all(np.isclose(r.notebook, x) for x in (r._3, r.dbt) if pd.notna(x)) for r in recon.itertuples()]
assert recon["match"].all(), recon
recon

,measure,notebook,Python demo,dbt,match
0,Ads kept,"1,167.00","1,167.00",NaN,True
1,Dropped: bad price,15.00,15.00,NaN,True
2,Dropped: outliers,18.00,18.00,NaN,True
3,Matched through typo correction,187.00,187.00,NaN,True
4,2023 Onix LTZ median price (R$),"68,400.00","68,400.00",NaN,True
5,"Median days to sell, below −6%",17.00,NaN,17.00,True
6,"Median days to sell, above +6%",30.00,NaN,30.00,True
7,Ads online above market,82.00,NaN,82.00,True
8,… total above their medians (R$),"530,500.00",NaN,"530,500.00",True


**Answer.** Cars priced more than 6% under their comparables sold in a median of 17 days; those more than 6% over took
30, and almost three in ten were still online. 82 ads still online sit above the market, R$ 530,500 above their
medians in total: that is the list a dealer works through first.

## Which library for interactive charts

| Library | Linked charts | Output | Best for |
|---|---|---|---|
| **Altair** (Vega-Lite), used here | Built in: selections and widgets shared across charts | Static HTML, no server | Reports that must open anywhere |
| **Plotly** | Per chart; linking across charts needs Dash | HTML; a Dash app for linking | Rich single charts, maps |
| **Bokeh** | Linked brushing through a shared data source | HTML, or a Bokeh server | Large scatter plots, custom tools |
| **HoloViews + Panel** | `link_selections` in one call | Notebook or Panel app | Turning a notebook into a small app |
| **Dash / Streamlit** | Any, through Python callbacks | Needs a server | Internal tools, like the InfoCars dealer screens |